# Golden-Case Smoke Check

This notebook is a smoke test against golden cases for the vulnrag RAG pipeline.
It talks to the **real Qdrant server** via `build_components()` (not an in-memory store).

**Prerequisites before running:**
- A running Qdrant instance (default: `localhost:6333`)
- A running Ollama instance with the configured embedding and chat models
- A synced vector store — run `scripts/run_sync.py` first to populate Qdrant with CVE data

The three golden cases check for a known-vulnerable version, a patched version, and a
completely unknown library, validating `vulnerable`, `safe`, and `not_found` verdicts.

In [ ]:
from vulnrag.factory import build_components
from vulnrag.query.answer import answer

# Notebooks talk to the REAL Qdrant server (via build_components), not :memory:.
store, emb, llm = build_components()

GOLDEN = [
    ("Безопасен ли log4j 2.14.1?", "vulnerable"),
    ("Is log4j 2.16.0 safe?", "safe"),
    ("Is totally-made-up-lib 1.0 safe?", "not_found"),
]
for q, expected in GOLDEN:
    r = answer(q, embedder=emb, store=store, llm=llm)
    print(q, "->", r.verdict, "(expected", expected, ")")
    print("  CVEs:", [c["cve_id"] for c in r.cves])